# PFE ML — Phase B : Optimisation des hyperparamètres des deux meilleures librairies

La Phase A a établi que HGB et CatBoost sont les deux meilleures librairies aux hyperparamètres par défaut (AP 0,155 / 0,148, précision top-0,1 % 84,9 % / 84,6 %). LightGBM et XGBoost sont écartés de la Phase B car ils étaient nettement en retard sur la métrique opérationnelle de la Phase A (précision top-0,1 %).

**Question de la Phase B :** Avec un budget de 20 configurations aléatoires × 3 plis de CV en walk-forward, une des deux librairies peut-elle améliorer son AP par rapport au réglage par défaut ?

## Conception méthodologique

- **Méthode de recherche :** `RandomizedSearchCV` avec `n_iter=20`. La recherche aléatoire est préférée à la recherche en grille pour les espaces de grande dimension, car à budget équivalent elle explore la surface de perte plus efficacement.
- **Stratégie de CV :** découpages temporels en walk-forward, **pas** de KFold aléatoire. Trois plis :
  - Pli 1 : train 2017–2020, validation 2021
  - Pli 2 : train 2017–2021, validation 2022
  - Pli 3 : train 2017–2022, validation 2023
  - 2024 est réservé comme jeu de test final mis de côté — jamais vu pendant l'optimisation.
- **Métrique d'évaluation :** `average_precision`. C'est la métrique de classement pour événements rares, et celle qu'utilise le résultat principal de la Phase A.
- **Données :** Le même échantillon hash déterministe de 2 M de lignes que les exécutions 1, 2, 7, 8, 9 et la Phase A. Comparaison strictement équivalente avec tout ce qui a précédé.
- **Déséquilibre de classes :** mécanisme natif de chaque librairie, fixé à la sémantique `balanced` — non optimisé. Le budget d'optimisation est réservé aux dimensions de régularisation et de capacité.
- **Matériel :** HGB s'optimise sur CPU (pas de support GPU). CatBoost s'optimise sur GPU (`task_type='GPU'`). L'entraînement final ré-exécute sur 2 M complets avec le même matériel que l'optimisation.

## Ce que produit ce notebook

Deux nouvelles entrées ajoutées à `model_run_comparison.csv` — `hgb_tuned` et `catboost_tuned` — aux côtés des 9 exécutions de référence et des 4 exécutions de la Phase A, pour **15 lignes comparables au total**. Plus deux fichiers JSON (`tuned_params_*.json`) contenant les meilleurs hyperparamètres pour la reproductibilité, et un graphique comparatif.

## Ce qui doit être présent sur la branche

Avant l'exécution, committez et poussez ceci sur `data-extraction` :
1. `app/tools/train_continuity_model.py` expose `--params-file PATH` (charge un JSON de kwargs du classifieur) et `--gpu` (chemin CUDA pour CatBoost / XGBoost).
2. `_build_model_pipeline()` accepte un dict de surcharge `extra_params` et un flag `gpu`.

## 1. Environnement d'exécution

**Choisissez un environnement GPU** (T4 ou A100 sur Pro). CPU seul fonctionne aussi, mais l'optimisation CatBoost prendra environ 4× plus de temps.

- L'optimisation HGB s'exécute sur CPU dans tous les cas (pas de support GPU dans HGB sklearn). Comptez ~70 min.
- L'optimisation CatBoost s'exécute sur GPU si disponible. Comptez ~20 min sur T4, ~10 min sur A100. Sur CPU il faudrait ~90 min.
- Total : ~90–100 min sur T4, ~80 min sur A100.

Si vous êtes pressé, passez `N_ITER` de 20 à 15 dans la section 4 pour économiser 25 % sur chaque librairie.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

START_YEAR = 2017
END_YEAR = 2024
TRAIN_MAX_ROWS = 2_000_000
TARGET = 'continuity_risk_12m_label'

N_ITER = 20            # configurations par librairie
N_SPLITS = 3           # plis de CV en walk-forward
TUNED_LIBRARIES = ['hgb', 'catboost']

FEATURES_GLOB = f'{DATA_LAKE}/features/company_year_features/**/*.parquet'
LABELS_GLOB = f'{DATA_LAKE}/features/risk_labels/**/*.parquet'

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR  =', BACKEND_DIR)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('TUNED LIBS   =', TUNED_LIBRARIES)
print('N_ITER       =', N_ITER, '/ library')
print('CV FOLDS     =', N_SPLITS, '(walk-forward)')

## 2. Mise à jour du code et installation des dépendances

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

In [ ]:
train_script_text = (Path(BACKEND_DIR) / 'app' / 'tools' / 'train_continuity_model.py').read_text(encoding='utf-8')
checks = {
    '--params-file CLI flag': '--params-file' in train_script_text,
    '--gpu CLI flag': '--gpu' in train_script_text,
    'extra_params parameter': 'extra_params: dict' in train_script_text,
    'gpu parameter on pipeline builder': 'gpu: bool' in train_script_text,
}
print('Phase B readiness:')
for label, ok in checks.items():
    print(f'  {"OK " if ok else "FAIL"}  {label}')
if not all(checks.values()):
    raise SystemExit('Phase B plumbing is missing on the pulled branch. Commit + push and re-run.')

import torch
gpu_available = torch.cuda.is_available()
print(f'\nGPU available: {gpu_available}')
if gpu_available:
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
else:
    print('Warning: GPU not detected. CatBoost will fall back to CPU and take ~4× longer.')

## 3. Chargement du même échantillon 2 M de lignes utilisé par toutes les autres exécutions

Reproduit exactement la requête de `train_continuity_model.py` : échantillon hash déterministe, ordonné temporellement pour le test 2024 mis de côté. Les mêmes `1 708 530` lignes d'entraînement et `291 470` lignes de test que la Phase A.

Lire l'intégralité du DataFrame de 2 M de lignes dans ce notebook (au lieu de sous-traiter le script d'entraînement pour chaque entraînement d'optimisation) nous évite de relire le parquet à chaque configuration de la recherche.

In [ ]:
import math, duckdb, pandas as pd, numpy as np
from app.tools.train_continuity_model import EXCLUDE_COLUMNS

filters = [f'l."{TARGET}" IS NOT NULL', f'f.prediction_year >= {START_YEAR}', f'f.prediction_year <= {END_YEAR}']
where_sql = ' AND '.join(filters)

con = duckdb.connect()
total_rows = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql}
""").fetchone()[0]
print(f'Eligible rows: {total_rows:,}')

modulus = 1_000_000
threshold = max(1, min(modulus, math.ceil((TRAIN_MAX_ROWS / total_rows) * modulus * 1.15)))
row_hash = "hash(CAST(f.siren AS VARCHAR) || ':' || CAST(f.prediction_year AS VARCHAR))"
feature_cols = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)"
).df()['column_name'].tolist()
select_cols = [c for c in feature_cols if c not in EXCLUDE_COLUMNS]
select_sql = ', '.join(f'f."{c}"' for c in select_cols)

df = con.execute(f"""
    SELECT {select_sql}, l."{TARGET}"
    FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql} AND {row_hash} % {modulus} < {threshold}
    ORDER BY {row_hash}
    LIMIT {TRAIN_MAX_ROWS}
""").df()
con.close()

print(f'Loaded shape: {df.shape}')
print(f'Class balance: {df[TARGET].value_counts().to_dict()}')
print(f'Years: {sorted(df["prediction_year"].unique())}')

In [ ]:
y = df[TARGET].astype(int)
feature_columns = [c for c in df.columns if c not in EXCLUDE_COLUMNS and c != TARGET]
X = df[feature_columns].copy()
for col in X.columns:
    if pd.api.types.is_bool_dtype(X[col]):
        X[col] = X[col].astype(float)

numeric_columns = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_columns = [c for c in X.columns if c not in numeric_columns]
print(f'Numeric columns:     {len(numeric_columns)}')
print(f'Categorical columns: {len(categorical_columns)} ({categorical_columns})')

test_year = int(X['prediction_year'].max())
train_mask = X['prediction_year'] < test_year
X_train, X_test = X[train_mask].reset_index(drop=True), X[~train_mask].reset_index(drop=True)
y_train, y_test = y[train_mask].reset_index(drop=True), y[~train_mask].reset_index(drop=True)
print(f'\nTrain (years < {test_year}): {len(X_train):,} rows, {y_train.sum():,} positives')
print(f'Test  (year = {test_year}):   {len(X_test):,} rows, {y_test.sum():,} positives')

## 4. Plis de CV en walk-forward et espaces de recherche

Des plis année par année donnent des estimations honnêtes de la généralisation temporelle. Un KFold sur lignes mélangées sous-estimerait la variance qui nous intéresse (dérive inter-années).

Les espaces de recherche sont volontairement étroits autour des valeurs par défaut de la Phase A. On ne ratisse pas large — on raffine autour d'une région connue comme bonne.

In [ ]:
from scipy.stats import loguniform

def walk_forward_folds(X_train_df, year_column='prediction_year', n_splits=3):
    years = sorted(X_train_df[year_column].unique())
    folds = []
    for i in range(len(years) - n_splits, len(years)):
        if i < 2:
            continue
        train_years = years[:i]
        val_year = years[i]
        train_idx = np.where(X_train_df[year_column].isin(train_years))[0]
        val_idx = np.where(X_train_df[year_column] == val_year)[0]
        folds.append((train_idx, val_idx))
        print(f'  fold {len(folds)}: train years {train_years}, val year {val_year}'
              f' ({len(train_idx):,} train rows, {len(val_idx):,} val rows)')
    return folds

print(f'Walk-forward folds ({N_SPLITS}):')
cv_folds = walk_forward_folds(X_train, n_splits=N_SPLITS)

hgb_param_dist = {
    'classifier__learning_rate': loguniform(0.01, 0.2),
    'classifier__max_leaf_nodes': [15, 31, 63, 127],
    'classifier__min_samples_leaf': [20, 50, 100, 200, 500],
    'classifier__l2_regularization': [0.0, 0.5, 1.0, 2.0, 5.0, 10.0],
    'classifier__max_features': [0.5, 0.7, 1.0],
}

catboost_param_dist = {
    'classifier__learning_rate': loguniform(0.01, 0.2),
    'classifier__depth': [4, 6, 8, 10],
    'classifier__l2_leaf_reg': [1, 3, 5, 10, 30],
    'classifier__bagging_temperature': [0.0, 0.5, 1.0, 2.0],
}

print('\nHGB search space:')
for k, v in hgb_param_dist.items():
    print(f'  {k}: {v}')
print('\nCatBoost search space:')
for k, v in catboost_param_dist.items():
    print(f'  {k}: {v}')

## 5. Optimisation HGB (CPU)

Comptez ~70 min. Chaque pli entraîne 20 configurations candidates, évaluées par précision moyenne.

In [ ]:
import json, time, importlib
from sklearn.model_selection import RandomizedSearchCV
# Recharge forcée du module d'entraînement pour récupérer toute modification
# du code source apportée par la section 2, même si le kernel a été démarré
# avant le git pull. Sans cela, le cache d'import de Python conserve l'ancien
# ``_build_model_pipeline`` et le ``clone()`` de sklearn dans
# ``RandomizedSearchCV`` peut échouer sur les contrôles d'identité des
# paramètres de constructeur de type liste (par ex. ``class_weights`` /
# ``cat_features`` de CatBoost) hérités du module pré-pull.
import app.tools.train_continuity_model as _tcm
importlib.reload(_tcm)
from app.tools.train_continuity_model import _build_model_pipeline

train_pos = int((y_train == 1).sum())
train_neg = int((y_train == 0).sum())

hgb_pipeline = _build_model_pipeline(
    family='hgb',
    categorical_columns=categorical_columns,
    train_positive_count=train_pos,
    train_negative_count=train_neg,
)

hgb_search = RandomizedSearchCV(
    hgb_pipeline,
    hgb_param_dist,
    n_iter=N_ITER,
    scoring='average_precision',
    cv=cv_folds,
    n_jobs=1,
    random_state=42,
    refit=False,
    return_train_score=False,
    verbose=2,
)

print(f'Starting HGB tuning: {N_ITER} configs x {len(cv_folds)} folds = {N_ITER * len(cv_folds)} fits')
start = time.time()
hgb_search.fit(X_train, y_train)
hgb_elapsed = time.time() - start
print(f'\nHGB tuning done in {hgb_elapsed/60:.1f} min')

hgb_best_params = {
    k.replace('classifier__', ''): (float(v) if isinstance(v, (np.floating,)) else v)
    for k, v in hgb_search.best_params_.items()
}
print(f'\nBest CV AP: {hgb_search.best_score_:.4f}')
print('Best params:')
for k, v in hgb_best_params.items():
    print(f'  {k}: {v}')

hgb_params_path = Path(DRIVE_ROOT) / 'ml-artifacts' / 'tuned_params_hgb.json'
hgb_params_path.parent.mkdir(parents=True, exist_ok=True)
hgb_params_path.write_text(json.dumps(hgb_best_params, indent=2), encoding='utf-8')
print(f'\nSaved: {hgb_params_path}')

## 6. Optimisation CatBoost (GPU)

Comptez ~20 min sur T4, ~10 min sur A100. Chaque pli entraîne 20 configurations candidates sur GPU.

In [ ]:
catboost_pipeline = _build_model_pipeline(
    family='catboost',
    categorical_columns=categorical_columns,
    train_positive_count=train_pos,
    train_negative_count=train_neg,
    gpu=gpu_available,
)

catboost_search = RandomizedSearchCV(
    catboost_pipeline,
    catboost_param_dist,
    n_iter=N_ITER,
    scoring='average_precision',
    cv=cv_folds,
    n_jobs=1,
    random_state=42,
    refit=False,
    return_train_score=False,
    verbose=2,
)

print(f'Starting CatBoost tuning ({"GPU" if gpu_available else "CPU"}): {N_ITER} configs x {len(cv_folds)} folds = {N_ITER * len(cv_folds)} fits')
start = time.time()
catboost_search.fit(X_train, y_train)
catboost_elapsed = time.time() - start
print(f'\nCatBoost tuning done in {catboost_elapsed/60:.1f} min')

catboost_best_params = {
    k.replace('classifier__', ''): (float(v) if isinstance(v, (np.floating,)) else v)
    for k, v in catboost_search.best_params_.items()
}
print(f'\nBest CV AP: {catboost_search.best_score_:.4f}')
print('Best params:')
for k, v in catboost_best_params.items():
    print(f'  {k}: {v}')

catboost_params_path = Path(DRIVE_ROOT) / 'ml-artifacts' / 'tuned_params_catboost.json'
catboost_params_path.write_text(json.dumps(catboost_best_params, indent=2), encoding='utf-8')
print(f'\nSaved: {catboost_params_path}')

## 7. Inspection de la surface de recherche

Avant de ré-entraîner les gagnants sur les 2 M complets, examinez les scores de CV pour vous assurer que la recherche n'a pas été dominée par une configuration aberrante isolée.

In [ ]:
def search_results_df(search, library):
    cv_results = pd.DataFrame(search.cv_results_)
    cv_results['library'] = library
    return cv_results[[
        'library', 'rank_test_score', 'mean_test_score', 'std_test_score',
        'params', 'mean_fit_time',
    ]].sort_values('rank_test_score').reset_index(drop=True)

hgb_results = search_results_df(hgb_search, 'hgb')
cb_results = search_results_df(catboost_search, 'catboost')

print('HGB — top 5 configurations:')
print(hgb_results.head(5).to_string(index=False))
print('\nCatBoost — top 5 configurations:')
print(cb_results.head(5).to_string(index=False))

combined_results = pd.concat([hgb_results, cb_results])
results_csv = Path(DRIVE_ROOT) / 'ml-artifacts' / 'tuning_search_results.csv'
combined_results.to_csv(results_csv, index=False)
print(f'\nSaved full search results to: {results_csv}')

In [ ]:
import matplotlib.pyplot as plt

tuning_dir = Path(DRIVE_ROOT) / 'ml-artifacts' / 'tuning_phase_b'
tuning_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (lib, results) in zip(axes, [('hgb', hgb_results), ('catboost', cb_results)]):
    scores = results['mean_test_score'].values
    stds = results['std_test_score'].values
    x = np.arange(len(scores))
    ax.errorbar(x, scores, yerr=stds, fmt='o', color='#0f766e', alpha=0.7)
    ax.axhline(scores.max(), color='#b91c1c', linestyle='--', label=f'best: {scores.max():.4f}')
    ax.set_title(f'{lib} — CV average precision per configuration')
    ax.set_xlabel('Configuration rank (best first)')
    ax.set_ylabel('CV mean AP (3-fold walk-forward)')
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(tuning_dir / 'tuning_score_distribution.png', dpi=160)
plt.show()

## 8. Ré-entraînement des gagnants sur les 2 M complets avec les paramètres optimisés

Invoque `train_continuity_model.py --params-file ... --model-family ...` pour chaque librairie afin que les exécutions optimisées atterrissent dans le pipeline d'artefacts standard (dossier `runs/*/`, ligne dans `model_run_comparison.csv`, `feature_importances.csv`, graphiques, etc.) aux côtés de toutes les exécutions précédentes.

Chaque ré-entraînement est un **unique entraînement** sur l'échantillon complet de 2 M avec les hyperparamètres optimisés — pas de CV à ce stade. C'est le jeu de test 2024 mis de côté lors des sections 3–4 qui sert à évaluer ces modèles.

In [ ]:
import shlex, subprocess, sys

retrain_results = {}
for family, params_path in [('hgb', hgb_params_path), ('catboost', catboost_params_path)]:
    print('=' * 70)
    print(f'Retraining {family} with tuned hyperparameters')
    print('=' * 70)
    cmd = [
        sys.executable, '-u',
        '-m', 'app.tools.train_continuity_model',
        '--data-lake-dir', f'{DRIVE_ROOT}/data-lake',
        '--artifacts-dir', f'{DRIVE_ROOT}/ml-artifacts',
        '--target', TARGET,
        '--train-start-year', str(START_YEAR),
        '--train-end-year', str(END_YEAR),
        '--max-rows', str(TRAIN_MAX_ROWS),
        '--min-rows', '1000',
        '--model-family', family,
        '--params-file', str(params_path),
    ]
    if family == 'catboost' and gpu_available:
        cmd.append('--gpu')
    print(' '.join(shlex.quote(p) for p in cmd))
    start = time.time()
    subprocess.run(cmd, check=True)
    elapsed = time.time() - start
    metadata = json.loads((Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_metadata.json').read_text(encoding='utf-8'))
    retrain_results[family] = {
        'run_name': metadata['run_name'],
        'run_dir': metadata['run_artifacts_dir'],
        'elapsed_seconds': elapsed,
        'metrics': metadata['metrics'],
    }
    m = metadata['metrics']
    print(
        f"\nDone ({elapsed:.0f}s). "
        f"AUC={m['roc_auc']:.4f}  "
        f"AP={m['average_precision']:.4f}  "
        f"F1@0.5={m['f1_at_0_5']:.4f}\n"
    )

print('Both tuned models retrained.')

## 9. Optimisé vs par défaut — côte à côte

In [ ]:
comparison_csv = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_run_comparison.csv'
all_runs = pd.read_csv(comparison_csv)

phase_a_runs = all_runs[all_runs['run_name'].str.contains('hgb_time-test-2024_cap-2m', na=False) |
                       all_runs['run_name'].str.contains('catboost_time-test-2024_cap-2m', na=False)].copy()
phase_a_runs = phase_a_runs.sort_values('trained_at').reset_index(drop=True)

summary_cols = [
    'run_name', 'model_family',
    'average_precision', 'roc_auc', 'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5',
]
summary_cols = [c for c in summary_cols if c in phase_a_runs.columns]
print('Phase A defaults + Phase B tuned (chronological):')
print(phase_a_runs[summary_cols].to_string(index=False))

def metric_delta(family, metric):
    rows = phase_a_runs[phase_a_runs['model_family'] == family]
    if len(rows) < 2:
        return None
    default = rows[metric].iloc[0]
    tuned = rows[metric].iloc[-1]
    return default, tuned, tuned - default

deltas = []
for family in TUNED_LIBRARIES:
    for metric in ['roc_auc', 'average_precision', 'f1_at_0_5']:
        result = metric_delta(family, metric)
        if result is None:
            continue
        default, tuned, delta = result
        deltas.append({
            'library': family, 'metric': metric,
            'default': default, 'tuned': tuned, 'delta': delta,
            'delta_pct': 100 * delta / default if default else None,
        })
deltas_df = pd.DataFrame(deltas)
print('\nTuned vs default deltas:')
print(deltas_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
metrics_to_plot = ['roc_auc', 'average_precision', 'f1_at_0_5']

for ax, metric in zip(axes, metrics_to_plot):
    width = 0.35
    x = np.arange(len(TUNED_LIBRARIES))
    defaults_vals = []
    tuned_vals = []
    for family in TUNED_LIBRARIES:
        rows = phase_a_runs[phase_a_runs['model_family'] == family].sort_values('trained_at')
        defaults_vals.append(rows[metric].iloc[0] if len(rows) > 0 else 0)
        tuned_vals.append(rows[metric].iloc[-1] if len(rows) > 1 else 0)
    ax.bar(x - width/2, defaults_vals, width, label='default', color='#94a3b8')
    ax.bar(x + width/2, tuned_vals, width, label='tuned', color='#0f766e')
    ax.set_xticks(x, TUNED_LIBRARIES)
    ax.set_title(metric)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    for xi, (d, t) in enumerate(zip(defaults_vals, tuned_vals)):
        ax.text(xi - width/2, d, f'{d:.4f}', ha='center', va='bottom', fontsize=8)
        ax.text(xi + width/2, t, f'{t:.4f}', ha='center', va='bottom', fontsize=8)
fig.suptitle('Phase B: tuned vs default per library')
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(tuning_dir / 'tuned_vs_default.png', dpi=160)
plt.show()

## 10. Affichage de la comparaison complète des 15 exécutions

In [ ]:
keep = [
    'run_name', 'model_family', 'rows', 'feature_count',
    'average_precision', 'roc_auc', 'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5',
]
keep = [c for c in keep if c in all_runs.columns]
print('All runs (chronological):')
print(all_runs[keep].to_string(index=False))
print(f'\nTotal runs: {len(all_runs)}')
print('Last two rows are the Phase B tuned models.')

## 11. Ce qu'il faut transmettre / Ce qu'il faut décider

### À joindre au mémoire
- `ml-artifacts/tuned_params_hgb.json`, `tuned_params_catboost.json` — hyperparamètres finaux pour la reproductibilité.
- `ml-artifacts/tuning_search_results.csv` — résultats complets de CV pour les 40 configurations (20 par librairie).
- `ml-artifacts/tuning_phase_b/tuning_score_distribution.png` — montre à quel point la surface de recherche est plate ou pointue par librairie. Surface plate = les valeurs par défaut étaient déjà bonnes ; surface pointue = l'optimisation a eu un effet réel.
- `ml-artifacts/tuning_phase_b/tuned_vs_default.png` — le graphique principal avant/après.
- Les deux dossiers `runs/*/` les plus récents — chacun contient `run_summary.md`, `feature_importances.csv` et les graphiques standards pour les modèles optimisés.

### À décider
Comparer les chiffres **optimisés** de la Phase B avec les chiffres **par défaut** de la Phase A (Run 9 = hgb par défaut, Run 13 = catboost par défaut) :
- Si l'AP optimisé améliore de ≥1 pp par rapport au défaut pour une librairie → l'optimisation valait le budget ; rapporter les chiffres optimisés comme finaux.
- Si l'AP optimisé améliore de <0,5 pp → les valeurs par défaut étaient déjà quasi-optimales ; rapporter cela comme un résultat méthodologique (« nous avons vérifié que le résultat aux valeurs par défaut était robuste à une perturbation des hyperparamètres »).
- Si l'AP optimisé *régresse* — la recherche a sur-appris sur la CV (peu probable avec 20 configs mais possible). À rapporter et investiguer.

### Formulation pour le mémoire (Phase B)
*« L'optimisation des hyperparamètres avec 20 configurations aléatoires et 3 plis de CV en walk-forward temporellement consciente a produit un modèle HGB optimisé avec AP X,XX (vs Y,YY par défaut, ΔZ pp) et un CatBoost optimisé avec AP X,XX (ΔZ pp). La stratégie de CV en walk-forward est adaptée au problème de prédiction décalé dans le temps, là où un KFold standard sous-estimerait la variance due à la dérive inter-années. Les deux librairies ont été optimisées avec des budgets comparables pour garantir une comparaison équitable. »*

Après la Phase B, les étapes naturelles suivantes sont la **Phase C** (stabilité temporelle — ré-entraîner le gagnant avec 2023 en test au lieu de 2024, vérifier si l'AUC reste dans la même plage) et la **Phase D** (interprétabilité par SHAP pour le gagnant).